# I2C device initialization

In [ ]:
from time import sleep as delay
from project.sc8583 import project

if "ic" in globals():
    ic.close()
    
ic = project(device="sc8583", revision="1p1", emulator="ch341", logging=False, is_gui=False)

# Equipments initialization

In [ ]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a, rigol_dp811a
from phy.power_analyzer import keysight_N6705
from phy.scope import tektronix_mdo34
from phy.battery_simulator import asd_906b

# from phy.eloader import sdl1030x
# from phy.relay_16ch import relay_box

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np
from time import sleep as delay
chart = plot()

dm1 = keithley_dm6500("USB0::0x05E6::0x6500::04651237::INSTR")
dm2 = keithley_dm6500("USB0::0x05E6::0x6500::04651251::INSTR")
# ps1 = rigol_dp821a()
# ps2 = rigol_dp811a()
ps = keysight_N6705()
ds = tektronix_mdo34()
bs = asd_906b(port=7)
# sm = keithley_2470()
# ld = sdl1030x()

# relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)

# --------------------------------------------------
# list_voltage = list(np.arange(5, 8, 0.005))
# voltage  = [round(num, 3) for num in list_voltage]
# --------------------------------------------------

from concurrent.futures import ThreadPoolExecutor

# Test code block

In [ ]:
ds.ch1.vertical_scale = 3
ds.ch2.vertical_scale = 3
ds.ch3.vertical_scale = 3
ds.ch4.vertical_scale = 3

ds.ch1.vertical_grid = -3
ds.ch2.vertical_grid = -3
ds.ch3.vertical_grid = -3
ds.ch4.vertical_grid = -3

ds.ch1.enable
ds.ch2.enable
ds.ch3.enable
ds.ch4.enable

In [ ]:
ds.ch1.label_off
ds.ch2.label_off
ds.ch3.label_off
ds.ch4.label_off

In [ ]:
ds.horizontal_center
ds.horizontal_scale = 200e-9

In [ ]:
ds.ch2.trigger_fall = 2

In [ ]:
ds.ch1.vertical_grid = 0
ds.ch2.vertical_grid = 0
ds.ch3.vertical_grid = 0
ds.ch4.vertical_grid = 0

In [ ]:
ds.ch1.vertical_scale = 0.05
ds.ch2.vertical_scale = 0.05
ds.ch3.vertical_scale = 0.05
ds.ch4.vertical_scale = 0.05

In [ ]:
ds.save_waveform = "VBAT=3.8V, IIN=2.2A - SAMSUNG 65W 3TO1"

In [ ]:
ds.save_waveform = "VBAT=3.8V, IIN=3.2A, TOOCKI 65W 2TO1"

In [ ]:
ds.save_waveform = "VCC during I2C access"

In [ ]:
ic.i2c_scan()

In [ ]:
ic.update_i2c_address(110)

In [ ]:
# --------------------------------------------------
list_voltage = list(np.arange(3, 25.1, 1))
vext_list  = [round(num, 1) for num in list_voltage]
# --------------------------------------------------

In [ ]:
ic.VEXT_OVP_DIS = 1
ic.NTC_FLT_DIS = 1
ic.ADC_RATE = 1
ic.ADC_FREEZE = 0

bs.vset = 4.5
delay(1)

for set_vext in vext_list:
    ps.ch1.vset = set_vext
    meas_vext = dm1.voltage_100
    meas_vext = dm1.voltage_100

    ic.ADC_EN = 1
    vext_adc = ic.VEXT_ADC
    vext_adc_conv = vext_adc * ic.lsb_vext
    accuracy = (vext_adc_conv - meas_vext) / meas_vext * 100
    print(f"{meas_vext:.06f} {vext_adc} {vext_adc:#06x} {vext_adc_conv:.05f} {accuracy:.05f}")

ps.ch1.vset = vext_list[0]